# Lecture 5 - Spurious Waffle Houses

McElreath's lectures for the whole book are available here: https://github.com/rmcelreath/statrethinking_winter2019

An R/Stan repo of code is available here: https://vincentarelbundock.github.io/rethinking2/

An excellent port to Python/PyMC Code is available here: https://github.com/aloctavodia/Statistical-Rethinking-with-Python-and-pymc

You are encouraged to work through both of these versions to re-enforce what we're doing in class.

In [ ]:
# Load R packages
library(cmdstanr)    # R interface to Stan
library(posterior)   # Working with posterior draws
library(bayesplot)   # Plotting posterior draws
library(MASS)        # kde2d() for 2D density contours

# Save the current figure to file (uncomment the savefig() calls below to use)
savefig <- function(file, width = 7, height = 7){
    dev.copy(jpeg, file, width = width, height = height, units = "in", res = 100)
    invisible(dev.off())
}

## Waffle Houses

Let's import the Waffe House devorce data:

In [ ]:
# Import data
ddata <- read.csv('WaffleDivorce.csv', sep = ';')
# Display top 5 rows
head(ddata, 5)

In [ ]:
# Table of descriptive statistics
summary(ddata)

So not unintense data to look at, but let's start with divorce and Waffle Houses

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 7)
plot(ddata$WaffleHouses, ddata$Divorce, col = "dodgerblue", pch = 16,
     xlab = 'Number of Waffle Houses', ylab = 'Divorce rate')
text(ddata$WaffleHouses, ddata$Divorce, ddata$Location, pos = 4, cex = 0.7)
abline(lm(Divorce ~ WaffleHouses, data = ddata))
# savefig('WaffleDivorce.jpg')

Or is divorce rate a product of marriage rate?

In [ ]:
plot(ddata$Marriage, ddata$Divorce, col = "dodgerblue", pch = 16,
     xlab = 'Marriage rate', ylab = 'Divorce rate')
text(ddata$Marriage, ddata$Divorce, ddata$Location, pos = 4, cex = 0.7)
abline(lm(Divorce ~ Marriage, data = ddata))
# savefig('WaffleMarriage.jpg')

Or age at marriage?

In [ ]:
plot(ddata$MedianAgeMarriage, ddata$Divorce, col = "dodgerblue", pch = 16,
     xlab = 'Median marriage age', ylab = 'Divorce rate')
text(ddata$MedianAgeMarriage, ddata$Divorce, ddata$Location, pos = 4, cex = 0.7)
abline(lm(Divorce ~ MedianAgeMarriage, data = ddata))
# savefig('WaffleAge.jpg')

And what does the South have to do with all this? Well with the assertion of a causal model we can take a look and see. For example, if we assert:

A->M->D
A->D

Then we can look and see what the affect of marriage rate (M) is on divorce (D), given that we know the median age (A). To do this we need a statistcal model to help evaluate this DAG.

$$
D_i \sim N(\mu_i,\sigma)\\
\mu_i = \beta_0+\beta_M M_i+\beta_A A_i
$$

There is nothing magic here - we've all done multiple regression before - but what is new is our causal assertion. Weird, that what we assert and assume changes things eh? But you should get very comfortable with this idea because it turns out it lies at the core of scientific enquiry - as Popper argued, causality is built consenually. 

First we should standardize variables:


In [ ]:
stdize <- function(x) (x - mean(x))/sd(x)

In [ ]:
A <- stdize(ddata$MedianAgeMarriage)
M <- stdize(ddata$Marriage)
D <- stdize(ddata$Divorce)

With covariates in hand we can do some prior predictive simulation to see what priors might look like in terms of possible lines:

In [ ]:
# Number of samples
nsamp <- 100
# Intercept
b0_ <- rnorm(nsamp, 0, .2)
# Marriage rate slope
bm_ <- rnorm(nsamp, 0, .5)
# Marriage age slope
ba_ <- rnorm(nsamp, 0, .5)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4)
par(mfrow = c(1, 2))

# Grab range of marriage ages to plot over
A_ <- seq(min(A), max(A), length.out = 50)
# Plot resulting lines given sample values for β0 and βa
plot(NA, xlim = range(A_), ylim = c(-3, 3), xlab = 'Median marriage age (std)', ylab = 'Divorce rate (std)')
for (i in 1:nsamp) lines(A_, b0_[i] + ba_[i]*A_, col = adjustcolor("black", 0.1))

# Grab range of marriage rates to plot over
M_ <- seq(min(M), max(M), length.out = 50)
# Plot resulting lines given sample values for β0 and βm
plot(NA, xlim = range(M_), ylim = c(-3, 3), xlab = 'Marriage rate (std)', ylab = 'Divorce rate (std)')
for (i in 1:nsamp) lines(M_, b0_[i] + bm_[i]*M_, col = adjustcolor("black", 0.1))
par(mfrow = c(1, 1))

Next, we can bulid a NUTS model in Stan. The `transformed parameters` block stores `Mu` for every state, just like a `pm.Deterministic` in PyMC:

In [ ]:
# Causal model

# Bayesian Stan
divorce_code <- "
data {
  int<lower=0> N;
  vector[N] A;   // median marriage age (std)
  vector[N] M;   // marriage rate (std)
  vector[N] D;   // divorce rate (std)
}
parameters {
  real Intercept;
  real Marriage_age;
  real Marriage_rate;
  real<lower=0> Sigma;
}
transformed parameters {
  // Linear model (identity link)
  vector[N] Mu = Intercept + Marriage_age*A + Marriage_rate*M;
}
model {
  // Priors
  Intercept ~ normal(0, .2);
  Marriage_age ~ normal(0, .5);
  Marriage_rate ~ normal(0, .5);
  Sigma ~ exponential(1);

  // Likelihood
  D ~ normal(Mu, Sigma);
}
"
divorce <- cmdstan_model(write_stan_file(divorce_code))

In [ ]:
# Run sampler
trace <- divorce$sample(data = list(N = length(D), A = A, M = M, D = D),
                        chains = 4, parallel_chains = 4, iter_sampling = 1000, refresh = 0, show_exceptions = FALSE)

In [ ]:
write.csv(t(trace$summary(c("Intercept", "Marriage_age"))), 'oop.csv')

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5)
mcmc_areas(trace$draws(c("Intercept", "Marriage_age", "Marriage_rate", "Sigma"))) + vline_0(linetype = 3)

The two single-predictor models (divorce on marriage rate only, and divorce on marriage age only) have exactly the same structure, so in Stan we only need to write and compile **one** simple regression model, then feed it different data:

In [ ]:
# Bayesian Stan - simple regression of y on a single predictor x
simple_code <- "
data {
  int<lower=0> N;
  vector[N] x;   // predictor
  vector[N] y;   // outcome
}
parameters {
  real Intercept;
  real Slope;
  real<lower=0> Sigma;
}
transformed parameters {
  // Linear model (identity link)
  vector[N] Mu = Intercept + Slope*x;
}
model {
  // Priors
  Intercept ~ normal(0, .2);
  Slope ~ normal(0, .5);
  Sigma ~ exponential(1);

  // Likelihood
  y ~ normal(Mu, Sigma);
}
"
simple_reg <- cmdstan_model(write_stan_file(simple_code))

In [ ]:
# Data for each model
divorce_m <- list(N = length(D), x = M, y = D)   # Divorce ~ Marriage rate
divorce_a <- list(N = length(D), x = A, y = D)   # Divorce ~ Marriage age

In [ ]:
# Run samplers
trace_m <- simple_reg$sample(data = divorce_m, chains = 4, parallel_chains = 4, refresh = 0, show_exceptions = FALSE)
trace_a <- simple_reg$sample(data = divorce_a, chains = 4, parallel_chains = 4, refresh = 0, show_exceptions = FALSE)

In [ ]:
# Helper to overlay the posterior densities of two sets of draws
plot_dist2 <- function(x1, x2, labels, main = ""){
    d1 <- density(x1); d2 <- density(x2)
    plot(d1, xlim = range(c(d1$x, d2$x)), ylim = c(0, max(c(d1$y, d2$y))), main = main, xlab = "", col = "dodgerblue", lwd = 2)
    lines(d2, col = "orange", lwd = 2)
    legend("topleft", legend = labels, col = c("dodgerblue", "orange"), lwd = 2, bty = "n", cex = 0.8)
}

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4)
par(mfrow = c(1, 2))

v_ <- 'Marriage age'
plot_dist2(as.vector(trace$draws("Marriage_age")), as.vector(trace_a$draws("Slope")),
           labels = paste(v_, c('(full)', '(only)')), main = v_)

v_ <- 'Marriage rate'
plot_dist2(as.vector(trace$draws("Marriage_rate")), as.vector(trace_m$draws("Slope")),
           labels = paste(v_, c('(full)', '(only)')), main = v_)
par(mfrow = c(1, 1))

# Plotting

Among the most - I'll say **the most** - important checks on your models is to plot the model and the data together. It is critical that you see things the way the model sees things, otherwise it is difficult to know how well you're doing in fitting these things. Three options are:

    1. Predictor residual plots
    2. Posterior prediction plots
    3. Counterfactual plots
    
Each has their own value and can tell us something about how our model is doing.

## 1. Predictor residual plots

There's an awful legacy in Biology of modelling the residuals of another model. It's awful because it's wrong, and you should never do it. It's wrong because it doesn't get the unceratinties right, prioritizing variation in the first analysis and hiding it in the second. This leads to biased estiamtes, possibly for both models, but certainly for the second. But there is some utility in seeing what information remains in one predictor when you already have information about the other (which is what multiple regression does). 

To do this we need to build individual models where we regress one predictor on the other, which will give us the marginal benefit of the other predictor conditional on knowing one of them.

So for the divorce case, we have two models:

In Stan these are again just our simple regression model, with different data:

In [ ]:
# Marriage age ~ Marriage rate
m_a <- list(N = length(A), x = M, y = A)

In [ ]:
# Marriage rate ~ Marriage age
a_m <- list(N = length(M), x = A, y = M)

In [ ]:
# Run samplers
trace_m <- simple_reg$sample(data = a_m, chains = 4, parallel_chains = 4, refresh = 0, show_exceptions = FALSE)
trace_a <- simple_reg$sample(data = m_a, chains = 4, parallel_chains = 4, refresh = 0, show_exceptions = FALSE)

In [ ]:
# Posterior mean of Mu for each state
colMeans(trace_a$draws("Mu", format = "matrix"))

In [ ]:
# Get residuals for the other predictor
m_pred <- colMeans(trace_m$draws("Mu", format = "matrix"))
residuals_m <- M - m_pred

a_pred <- colMeans(trace_a$draws("Mu", format = "matrix"))
residuals_a <- A - a_pred

In [ ]:
median(trace_a$draws("Intercept"))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4)
par(mfrow = c(1, 2))

xnew <- seq(-2, 3, length.out = 100)

# Marriage age ~ Marriage rate
plot(M, A, col = "dodgerblue", pch = 16, xlab = 'Marriage rate', ylab = '')
title(ylab = 'Marriage age', col.lab = 'red')
lines(xnew, median(trace_a$draws("Intercept")) + median(trace_a$draws("Slope"))*xnew)
segments(M, a_pred, M, a_pred + residuals_a, col = 'grey')

# Marriage rate ~ Marriage age
plot(A, M, col = "dodgerblue", pch = 16, xlab = '', ylab = 'Marriage rate')
title(xlab = 'Marriage age', col.lab = 'red')
lines(xnew, median(trace_m$draws("Intercept")) + median(trace_m$draws("Slope"))*xnew)
segments(A, m_pred, A, m_pred + residuals_m, col = 'grey')
par(mfrow = c(1, 1))

What's seemingly bonkers, is that we now have the residuals for each parameter, we can plot them against divorce to see how the **full model** actually sees these things inside their guts:

In [ ]:
as.vector(trace$draws("Intercept"))

In [ ]:
as.vector(trace$draws("Intercept"))[1]

In [ ]:
par(mfrow = c(1, 2))

xnew <- seq(-2, 3, length.out = 100)
# Posterior draws from the full model
post <- as_draws_df(trace$draws(c("Intercept", "Marriage_age", "Marriage_rate")))

v_ <- 'Marriage_age'
plot(residuals_a, D, col = "dodgerblue", pch = 16, xlab = 'Marriage age', ylab = 'Divorce rate')
for (i in 1:100) lines(xnew, post$Intercept[i] + post[[v_]][i]*xnew, col = adjustcolor("black", 0.05))
lines(xnew, median(post$Intercept) + median(post[[v_]])*xnew, col = "dodgerblue", lwd = 2)

v_ <- 'Marriage_rate'
plot(residuals_m, D, col = "dodgerblue", pch = 16, xlab = 'Marriage rate', ylab = 'Divorce rate')
for (i in 1:100) lines(xnew, post$Intercept[i] + post[[v_]][i]*xnew, col = adjustcolor("black", 0.05))
lines(xnew, median(post$Intercept) + median(post[[v_]])*xnew, col = "dodgerblue", lwd = 2)
par(mfrow = c(1, 1))

So conditional on knowing marriage rate, marriage age still tells us something useful about divorce, but conditional on knowing marriage age, marriage rate tells us very little. Hence the difference in parameter estimates, with marriage age having a way bigger effect size. 

Incidentally, while we have these residuals, let's take a look at their distribtuion and what they mean:

In [ ]:
par(mfrow = c(1, 2))

# Residuals of marriage age (from the Marriage age ~ Marriage rate model)
tmp <- hist(residuals_a, main = "", xlab = "residuals_a")
lines(xnew, dnorm(xnew, 0, mean(trace_a$draws("Sigma")))*max(tmp$counts)/dnorm(0, 0, mean(trace_a$draws("Sigma"))))

# Residuals of marriage rate (from the Marriage rate ~ Marriage age model)
tmp <- hist(residuals_m, main = "", xlab = "residuals_m")
lines(xnew, dnorm(xnew, 0, mean(trace_m$draws("Sigma")))*max(tmp$counts)/dnorm(0, 0, mean(trace_m$draws("Sigma"))))
par(mfrow = c(1, 1))

The distribution of the residuals is the error distribution (`Sigma`) for the linear model - i.e. `Sigma` describes the magnitude of the deviations from the regression line. 

## 2. Posterior prediction plots

Another important question is - how well is our model capturing the observed data? Are our predictions about each observation any good? Having used MCMC for our inference (and stored the values of `Mu` in the `transformed parameters` block), we can just grab the observed and expected values and plot them:

In [ ]:
head(as_draws_df(trace$draws("Mu")))

In [ ]:
# Matrix of posterior draws (rows) for each state observation (columns)
PostObs <- trace$draws("Mu", format = "matrix")
colnames(PostObs) <- ddata$Location
head(PostObs)

In [ ]:
# Calculate expected y values and UI's
y <- apply(PostObs, 2, median)
y_l95 <- apply(PostObs, 2, quantile, probs = 0.025)
y_u95 <- apply(PostObs, 2, quantile, probs = 0.975)

In [ ]:
# Plot expected vs observed
options(repr.plot.width = 7, repr.plot.height = 5)
plot(D, y, col = "dodgerblue", pch = 16, ylim = range(c(y_l95, y_u95)), xlab = "Observed divorce (std)", ylab = "Predicted divorce (std)")
segments(D, y_l95, D, y_u95, col = 'grey')
abline(0, 1, lty = 3)

So we can see that our model underpredicts high divorce rates (right side) and overpredicts low devorce rates (left side) but that is to be expected, it is a normal model after all an predictions tend to shrink toward the overall average. 

But it does look like there are some outlying values, let's label a few

In [ ]:
# Plot expected vs observed
plot(D, y, col = "dodgerblue", pch = 16, ylim = range(c(y_l95, y_u95)), xlab = "Observed divorce (std)", ylab = "Predicted divorce (std)")
segments(D, y_l95, D, y_u95, col = 'grey')
abline(0, 1, lty = 3)

# Label states that are >x SD off
x <- 1.3
off <- abs(D - y) > x
text(D[off], y[off], ddata$Location[off], pos = 4)

## 3. Counterfactual plots

Counterfactuals are frequently brought up in statistical circles, and especially in economics, as a device to imagine what would happen if something else had happened in our data. In the case of counterfactual plots, they show us what happens if we manipulate one variable while keeping the others constant. 

If we return to the causal model where median marriage age influences divorce rate both directly and indirectly via marriage rate, we can develop a counterfactual plot by simulating from our `divorce` and `a_m` models above.

In [ ]:
# Divorce model trace
trace$summary(c("Intercept", "Marriage_rate", "Marriage_age", "Sigma"))

In [ ]:
# a_m model trace (Marriage rate ~ Marriage age)
trace_m$summary(c("Intercept", "Slope", "Sigma"))

With these values in place, we can see what the predicted change in divorce rate is across the full range of changes in median marriage age. To do this, we first choose the range of marriage ages:

In [ ]:
# Marriage age prediction range
nsim <- 100
A_new <- seq(min(A), max(A), length.out = nsim)

Next we calculate the expected effect of marriage age on marriage rate:

In [ ]:
# Marriage rates given marriage age range
M_new <- median(trace_m$draws("Intercept")) + median(trace_m$draws("Slope"))*A_new

And finally we simulate from the full `divorce` model, given our new (counterfactual) covariate values:


In [ ]:
D_new <- median(trace$draws("Intercept")) + median(trace$draws("Marriage_age"))*A_new + median(trace$draws("Marriage_rate"))*M_new

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4)
par(mfrow = c(1, 2))

plot(A_new, D_new, type = "l", lwd = 2, xlab = 'Marriage age', ylab = 'Divorce rate')

plot(A_new, M_new, type = "l", lwd = 2, xlab = 'Marriage age', ylab = 'Marriage rate')
par(mfrow = c(1, 1))

# Masked relationships

One of the many (many, many,...) pitfalls of statistical models is the presence of masked relationships - variables that counteract each other so they each appear to have no particular relationship. The primate milk data has just such a case.

In [ ]:
# Import data
mdata <- read.csv('milk.csv', sep = ';')
# Drop rows where neocortex percent is NA
mdata <- mdata[!is.na(mdata$neocortex.perc), ]
# Add log(mass) column
mdata$log_mass <- log(mdata$mass)
head(mdata)

If we take a look at the bivariate relationships among variables, it seems there's not too much going on beyond the relationship between 

In [ ]:
# Pairs plot: scatter above the diagonal, 2D density contours below, densities on the diagonal
panel_kde <- function(x, y, ...){
    k <- kde2d(x, y, n = 50)
    contour(k, add = TRUE, drawlabels = FALSE, col = "dodgerblue")
}
panel_dens <- function(x, ...){
    usr <- par("usr"); on.exit(par(usr = usr))
    d <- density(x)
    par(usr = c(usr[1:2], 0, max(d$y)*1.1))
    lines(d, lwd = 2, col = "dodgerblue")
}
options(repr.plot.width = 7, repr.plot.height = 7)
pairs(mdata[, c('kcal.per.g', 'log_mass', 'neocortex.perc')],
      upper.panel = function(x, y, ...) points(x, y, pch = 16, col = "dodgerblue"),
      lower.panel = panel_kde, diag.panel = panel_dens)

Yet if we run the full model for the relationship between log(mass) and neocortex.conc on kcal.per.g, we get a surprise:

In [ ]:
# Grab variables of interest
logMass <- stdize(mdata$log_mass)
neocorp <- stdize(mdata$neocortex.perc)
kcal <- stdize(mdata$kcal.per.g)

In [ ]:
# Bayesian Stan
milker_code <- "
data {
  int<lower=0> N;
  vector[N] logMass;
  vector[N] neocorp;
  vector[N] kcal;
}
parameters {
  real Intercept;
  real log_mass;
  real neocortex_perc;
  real<lower=0> Sigma;
}
model {
  // Priors
  Intercept ~ normal(0, .2);
  log_mass ~ normal(0, .5);
  neocortex_perc ~ normal(0, .5);
  Sigma ~ exponential(1);

  // Linear model (identity link)
  vector[N] mu = Intercept + log_mass*logMass + neocortex_perc*neocorp;

  // Likelihood
  kcal ~ normal(mu, Sigma);
}
"
milker <- cmdstan_model(write_stan_file(milker_code))

In [ ]:
trace_milk <- milker$sample(data = list(N = length(kcal), logMass = logMass, neocorp = neocorp, kcal = kcal),
                            chains = 4, parallel_chains = 4, iter_sampling = 1000, refresh = 0, show_exceptions = FALSE)

In [ ]:
trace_milk$summary()

What's this now? Both log(mass) and percent neocortex do not span zero, meaning they have strong relationships in the data. This can happen and is due to some unknown variable having synnergistic effects on both variables, but in different directions. Because they both happen they appear to not have any effect in a bivariate plot, but when both are present, their actual effects are revealed. Knowing that it can happen is half the battle. But it still sucks that it does.